# Week 5: Apache Spark - Data Cleaning, Transformation and Aggregation
Objective
Understand Spark fundamentals and perform data cleaning, transformation, and aggregation using DataFrames.

Dataset: Kaggle - Retail Analysis Large Dataset

Column mapping used below (assignment terms -> actual columns):
- user_id -> Customer_ID
- transaction_date -> Date
- region -> Country
- product_category -> Product_Category
- sale_amount -> Total_Amount
- price -> Amount
- subscription -> Customer_Segment
- status -> Order_Status
- store_id -> City
- username -> Name

In [39]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [40]:
spark = SparkSession.builder.appName("Week5_Spark_Assignment").getOrCreate()
print("Spark Session Created Successfully")

Spark Session Created Successfully


In [41]:
df = spark.read.csv("DataSet/retail_data.csv", header=True, inferSchema=True)
print("Data set readed")

Data set readed


In [42]:
df.show(5)

+--------------+-----------+-------------------+-------------------+-------------+--------------------+----------+---------------+-------+---------+----+------+------+----------------+----------+------+---------+-------------------+---------------+-----------+------------+----------------+-------------+------------+---------+---------------+--------------+------------+-------+-----------------+
|Transaction_ID|Customer_ID|               Name|              Email|        Phone|             Address|      City|          State|Zipcode|  Country| Age|Gender|Income|Customer_Segment|      Date|  Year|    Month|               Time|Total_Purchases|     Amount|Total_Amount|Product_Category|Product_Brand|Product_Type| Feedback|Shipping_Method|Payment_Method|Order_Status|Ratings|         products|
+--------------+-----------+-------------------+-------------------+-------------+--------------------+----------+---------------+-------+---------+----+------+------+----------------+----------+------+--

In [43]:
df.printSchema()

root
 |-- Transaction_ID: double (nullable = true)
 |-- Customer_ID: double (nullable = true)
 |-- Name: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- Phone: double (nullable = true)
 |-- Address: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Zipcode: double (nullable = true)
 |-- Country: string (nullable = true)
 |-- Age: double (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Income: string (nullable = true)
 |-- Customer_Segment: string (nullable = true)
 |-- Date: string (nullable = true)
 |-- Year: double (nullable = true)
 |-- Month: string (nullable = true)
 |-- Time: timestamp (nullable = true)
 |-- Total_Purchases: double (nullable = true)
 |-- Amount: double (nullable = true)
 |-- Total_Amount: double (nullable = true)
 |-- Product_Category: string (nullable = true)
 |-- Product_Brand: string (nullable = true)
 |-- Product_Type: string (nullable = true)
 |-- Feedback: string (nullable =

In [44]:
print("Total Rows:", df.count())
print("Total Columns:", len(df.columns))

Total Rows: 302010
Total Columns: 30


In [45]:
df.describe().show()

+-------+------------------+------------------+-----------+------------------+-------------------+--------------------+---------+-------+-----------------+---------+------------------+------+------+----------------+--------+------------------+---------+-----------------+------------------+------------------+----------------+-------------+------------+--------+---------------+--------------+------------+------------------+--------+
|summary|    Transaction_ID|       Customer_ID|       Name|             Email|              Phone|             Address|     City|  State|          Zipcode|  Country|               Age|Gender|Income|Customer_Segment|    Date|              Year|    Month|  Total_Purchases|            Amount|      Total_Amount|Product_Category|Product_Brand|Product_Type|Feedback|Shipping_Method|Payment_Method|Order_Status|           Ratings|products|
+-------+------------------+------------------+-----------+------------------+-------------------+--------------------+---------+-

# Assignment Questions

# Q1: What are the key limitations of traditional MapReduce that make Spark a preferred choice for modern big data processing?

Limitations of MapReduce:
- Disk-based processing between every stage
- Slow for iterative jobs
- High latency, complex programming model
- No native support for streaming/ML

Spark advantages: in-memory computing, faster execution, easy DataFrame API.

# Q2: Explain how Spark uses In-Memory Computing to speed up iterative machine learning algorithms compared to disk-based systems.

- Disk-based systems read/write data from disk in every iteration.
- Spark loads data once and keeps it in memory, so repeated steps (like ML training) are much faster.
- Less disk I/O means better performance for iterative algorithms.

# Q3: Write a code snippet to remove all duplicate rows from a DataFrame based on a specific set of columns: user_id and transaction_date.

In [46]:
df_no_duplicates = df.dropDuplicates(["Customer_ID", "Date"])

print("Rows Before:", df.count())
print("Rows After:", df_no_duplicates.count())

df_no_duplicates.show(5)

Rows Before: 302010


Rows After: 298196


+--------------+-----------+-------------+--------------------+-------------+--------------------+--------------+---------------+-------+---------+----+------+------+----------------+----------+------+--------+-------------------+---------------+-----------+------------+----------------+-------------+------------+--------+---------------+--------------+------------+-------+-------------+
|Transaction_ID|Customer_ID|         Name|               Email|        Phone|             Address|          City|          State|Zipcode|  Country| Age|Gender|Income|Customer_Segment|      Date|  Year|   Month|               Time|Total_Purchases|     Amount|Total_Amount|Product_Category|Product_Brand|Product_Type|Feedback|Shipping_Method|Payment_Method|Order_Status|Ratings|     products|
+--------------+-----------+-------------+--------------------+-------------+--------------------+--------------+---------------+-------+---------+----+------+------+----------------+----------+------+--------+--------

# Q4: Given a DataFrame df_sales, filter for region = 'West', then group by product_category to find the average sale_amount.

In [47]:
df_sales = df

# region -> Country in this dataset, change 'USA' to a value present in your data
avg_sales = df_sales.filter(col("Country") == "USA") \
    .groupBy("Product_Category") \
    .agg(avg("Total_Amount").alias("avg_sale_amount"))

avg_sales.show()

+----------------+------------------+
|Product_Category|   avg_sale_amount|
+----------------+------------------+
|            NULL| 1359.646200994023|
|         Grocery|1360.6844003757878|
|     Electronics|1364.5333085856948|
|        Clothing|1364.8149346954651|
|           Books| 1369.030714929882|
|      Home Decor| 1362.217525025326|
+----------------+------------------+



# Q5: What is the difference between .na.drop() and .na.fill()? Provide a code example of filling null values in a status column with the string 'Unknown'.

- .na.drop() removes rows that have null values.
- .na.fill() fills the null values with a given value instead of removing the row.

In [48]:
df_status = df.na.fill({"Order_Status": "Unknown"})
df_status.select("Customer_ID", "Order_Status").show(5)

+-----------+------------+
|Customer_ID|Order_Status|
+-----------+------------+
|    37249.0|     Shipped|
|    69749.0|  Processing|
|    30192.0|  Processing|
|    62101.0|  Processing|
|    27901.0|     Shipped|
+-----------+------------+
only showing top 5 rows


# Q6: Write a query to find the total count of records for each city in a DataFrame, but only for cities where the count is greater than 100.

In [49]:
city_counts = df.groupBy("City").agg(count("*").alias("record_count"))

city_counts_filtered = city_counts.filter(col("record_count") > 100)

city_counts_filtered.show()

+------------+------------+
|        City|record_count|
+------------+------------+
|     Hanover|        2291|
|    Winnipeg|        2406|
|      Cairns|        2242|
|     Kelowna|        2195|
|    Brighton|        2279|
|     Bendigo|        2316|
|    Canberra|        2202|
|      Ottawa|        2204|
|   Edinburgh|        2317|
|  Manchester|        2267|
|    Adelaide|        2221|
|   Frankfurt|       10182|
|        Hull|        2225|
|Philadelphia|         840|
|      Oxford|        2242|
|     Glasgow|        2193|
|     Cardiff|        2261|
|      Bochum|        2208|
|  Wollongong|        2262|
|   Cleveland|         885|
+------------+------------+
only showing top 20 rows


# Q7: How does the immutability of Spark DataFrames affect how you perform data cleaning steps like dropping columns or renaming them?

- Spark DataFrames cannot be changed in place, every operation creates a new DataFrame.
- So we need to store the result in a variable, like df = df.drop("column_name").
- The original DataFrame remains unchanged unless we overwrite it.

In [50]:
df_dropped = df.drop("Feedback")
df_renamed = df_dropped.withColumnRenamed("Total_Amount", "total_sale")

df_renamed.show(5)

+--------------+-----------+-------------------+-------------------+-------------+--------------------+----------+---------------+-------+---------+----+------+------+----------------+----------+------+---------+-------------------+---------------+-----------+-----------+----------------+-------------+------------+---------------+--------------+------------+-------+-----------------+
|Transaction_ID|Customer_ID|               Name|              Email|        Phone|             Address|      City|          State|Zipcode|  Country| Age|Gender|Income|Customer_Segment|      Date|  Year|    Month|               Time|Total_Purchases|     Amount| total_sale|Product_Category|Product_Brand|Product_Type|Shipping_Method|Payment_Method|Order_Status|Ratings|         products|
+--------------+-----------+-------------------+-------------------+-------------+--------------------+----------+---------------+-------+---------+----+------+------+----------------+----------+------+---------+--------------

# Q8: Write a Spark command to filter a dataset for rows where the age is between 18 and 30 (inclusive) and the subscription is 'Premium'.

In [51]:
df_filtered = df.filter((col("Age") >= 18) & (col("Age") <= 30) & (col("Customer_Segment") == "Premium"))

df_filtered.show(10)

+--------------+-----------+---------------+--------------------+-------------+--------------------+----------+-------+-------+-------+----+------+------+----------------+----------+------+---------+-------------------+---------------+-----------+------------+----------------+-------------+------------+---------+---------------+--------------+------------+-------+-----------------+
|Transaction_ID|Customer_ID|           Name|               Email|        Phone|             Address|      City|  State|Zipcode|Country| Age|Gender|Income|Customer_Segment|      Date|  Year|    Month|               Time|Total_Purchases|     Amount|Total_Amount|Product_Category|Product_Brand|Product_Type| Feedback|Shipping_Method|Payment_Method|Order_Status|Ratings|         products|
+--------------+-----------+---------------+--------------------+-------------+--------------------+----------+-------+-------+-------+----+------+------+----------------+----------+------+---------+-------------------+-----------

# Q9: When cleaning a dataset, why is it often better to handle null values before performing mathematical aggregations like sum() or avg()?

- sum() and avg() skip null values automatically, which can give a wrong picture of the data.
- If we fill or drop nulls first, the aggregation result is more accurate and consistent.

# Q10: Write the code to revise a column named raw_timestamp by casting it to a TimestampType and renaming it to event_time.

In [52]:
df_ts = df.withColumn("Date", to_timestamp(col("Date"), "M/d/yyyy"))
df_ts = df_ts.withColumnRenamed("Date", "event_time")

df_ts.select("Customer_ID", "event_time").show(5)

+-----------+-------------------+
|Customer_ID|         event_time|
+-----------+-------------------+
|    37249.0|2023-09-18 00:00:00|
|    69749.0|2023-12-31 00:00:00|
|    30192.0|2023-04-26 00:00:00|
|    62101.0|2023-05-08 00:00:00|
|    27901.0|2024-01-10 00:00:00|
+-----------+-------------------+
only showing top 5 rows


# Q11: Explain the Shuffle process that occurs during a grouping operation. Why is it considered a wide transformation?

- Shuffle means moving data across different partitions/nodes so that same keys come together.
- groupBy needs all rows of the same key on one node, so Spark shuffles the data - this makes it a wide transformation, unlike filter/select which work on data already present in the partition.

# Q12: Write a code snippet that identifies and removes rows where the email column contains null values OR the username is an empty string.

In [53]:
clean_df = df.filter(col("Email").isNotNull())
clean_df = clean_df.filter(col("Name") != "")

clean_df.show(5)

+--------------+-----------+-------------------+-------------------+-------------+--------------------+----------+---------------+-------+---------+----+------+------+----------------+----------+------+---------+-------------------+---------------+-----------+------------+----------------+-------------+------------+---------+---------------+--------------+------------+-------+-----------------+
|Transaction_ID|Customer_ID|               Name|              Email|        Phone|             Address|      City|          State|Zipcode|  Country| Age|Gender|Income|Customer_Segment|      Date|  Year|    Month|               Time|Total_Purchases|     Amount|Total_Amount|Product_Category|Product_Brand|Product_Type| Feedback|Shipping_Method|Payment_Method|Order_Status|Ratings|         products|
+--------------+-----------+-------------------+-------------------+-------------+--------------------+----------+---------------+-------+---------+----+------+------+----------------+----------+------+--

# Q13: How do you use the .agg() function to calculate multiple statistics at once, such as the min, max, and mean of the price column?

In [54]:
df.agg(
    min("Amount").alias("min_price"),
    max("Amount").alias("max_price"),
    avg("Amount").alias("mean_price")
).show()

+-----------+----------+------------------+
|  min_price| max_price|        mean_price|
+-----------+----------+------------------+
|10.00021923|499.997911|255.16365896284515|
+-----------+----------+------------------+



# Q14: In the context of cleaning a dataset, what is the risk of using inferSchema=true when your source data contains messy or inconsistent date formats?

- Spark may guess the wrong data type for the column.
- Some dates may not match the guessed format and become null.
- This can lead to wrong analysis later.

# Q15: Write a final processing pipeline that: 1. Filters out duplicates. 2. Fills null prices with 0. 3. Groups by store_id to calculate total revenue.

In [55]:
final_df = df.dropDuplicates()
final_df = final_df.na.fill({"Amount": 0})

revenue = final_df.groupBy("City").agg(sum("Amount").alias("total_revenue"))

revenue.show()

+------------+------------------+
|        City|     total_revenue|
+------------+------------------+
|     Hanover| 597803.5246248101|
|    Winnipeg| 606079.7406111599|
|     Phoenix|242729.47866608002|
|      Cairns|   570274.69274849|
|    Brighton|   579032.42796672|
|     Kelowna|   547270.90061022|
|       Omaha|   226576.40300056|
|     Bendigo|   594943.30061632|
|    Canberra|   563062.32804283|
|      Ottawa|   567426.92017913|
|   Edinburgh|   593679.22190875|
|      Dallas|231180.09015722002|
|  Manchester|   575683.26984669|
|     Oakland|232561.79244172003|
|    Adelaide|   560725.12584795|
|   Frankfurt| 2586755.603091089|
|        Hull| 563229.7375179399|
| San Antonio|246837.29836041998|
|     Raleigh|228001.94075550005|
|Philadelphia|   219149.95522564|
+------------+------------------+
only showing top 20 rows


In [ ]:
# Save cleaned dataset
cleaned_df = df.dropDuplicates()
cleaned_df = cleaned_df.na.fill({"Amount": 0})

cleaned_df.toPandas().to_csv("cleaned_dataset.csv", index=False)
print("Cleaned dataset saved successfully.")

/home/manas/.local/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/home/manas/.local/lib/python3.12/site-packages/pyspark/sql/pandas/types.py:778: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
